# NB01: MAG Extraction & Quality Control

Extract per-bin FASTA files from ENIGMA CORAL metagenome assemblies for the 623 MAGs,
then run CheckM2 for quality filtering.

**Must run on JupyterHub** — assembly FASTAs are at cluster-local paths
(`/h/jmc/data/enigma/metagenomics/`).

In [ ]:
import os, json, gzip
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session()
DATA_DIR = Path('../data')
MAG_DIR = DATA_DIR / 'mag_fastas'
MAG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load MAG Bin Metadata from CORAL

In [ ]:
bins = spark.sql('''
    SELECT 
        b.assembly_id, b.bin_name, b.contig_ids,
        b.completeness, b.contamination, b.taxonomy,
        a.assembly_name, a.sample_name
    FROM enigma_coral.sdt_bin b
    JOIN enigma_coral.sdt_assembly a ON b.assembly_id = a.id
''').toPandas()

print(f'{len(bins)} total bins')
print(f'{bins["assembly_id"].nunique()} assemblies')
print(f'Completeness: mean={bins["completeness"].mean():.1f}%, median={bins["completeness"].median():.1f}%')
print(f'Contamination: mean={bins["contamination"].mean():.1f}%, median={bins["contamination"].median():.1f}%')
bins.head()

## 2. Quality Filter (completeness ≥50%, contamination ≤10%)

In [ ]:
hq_bins = bins[
    (bins['completeness'] >= 50) & 
    (bins['contamination'] <= 10)
].copy()

print(f'{len(hq_bins)}/{len(bins)} bins pass QC (completeness>=50%, contamination<=10%)')
print(f'{bins["completeness"].isna().sum()} bins have no completeness value')

dropped = bins[~bins.index.isin(hq_bins.index)]
print(f'\nDropped {len(dropped)} bins:')
print(f'  Low completeness: {(dropped["completeness"] < 50).sum()}')
print(f'  High contamination: {(dropped["contamination"] > 10).sum()}')
print(f'  Missing QC values: {dropped[["completeness","contamination"]].isna().any(axis=1).sum()}')

## 3. Locate Assembly FASTA Files

In [ ]:
ASSEMBLY_BASE = Path('/h/jmc/data/enigma/metagenomics')

assemblies = spark.sql('''
    SELECT id, assembly_name, sample_name, file_path
    FROM enigma_coral.sdt_assembly
''').toPandas()

print(f'{len(assemblies)} assemblies in CORAL')
print(f'\nAssembly names:')
for _, row in assemblies.iterrows():
    fp = row['file_path'] if pd.notna(row['file_path']) else 'NULL'
    exists = Path(fp).exists() if pd.notna(row['file_path']) else False
    print(f'  {row["assembly_name"]}: {fp} [{"OK" if exists else "MISSING"}]')

In [ ]:
# Try to discover FASTA files if file_path column is NULL
# Common patterns: {assembly_name}.fasta, {assembly_name}.fa, {assembly_name}.fna,
#                  {sample_name}/assembly.fasta, etc.

assembly_fastas = {}
search_dirs = [
    ASSEMBLY_BASE,
    ASSEMBLY_BASE / 'assemblies',
    ASSEMBLY_BASE / 'contigs',
]

for _, row in assemblies.iterrows():
    aid = row['id']
    aname = row['assembly_name']
    
    # First try the file_path from the database
    if pd.notna(row['file_path']) and Path(row['file_path']).exists():
        assembly_fastas[aid] = Path(row['file_path'])
        continue
    
    # Search for FASTA files matching assembly name
    found = False
    for sdir in search_dirs:
        if not sdir.exists():
            continue
        for ext in ['.fasta', '.fa', '.fna', '.fasta.gz', '.fa.gz', '.fna.gz']:
            candidate = sdir / f'{aname}{ext}'
            if candidate.exists():
                assembly_fastas[aid] = candidate
                found = True
                break
        if found:
            break
    
    if not found:
        # Try looking in a subdirectory named after the assembly
        subdir = ASSEMBLY_BASE / aname
        if subdir.exists():
            for f in subdir.glob('*contigs*'):
                assembly_fastas[aid] = f
                found = True
                break

print(f'Found FASTAs for {len(assembly_fastas)}/{len(assemblies)} assemblies')
for aid, fp in assembly_fastas.items():
    aname = assemblies[assemblies['id'] == aid]['assembly_name'].values[0]
    size_mb = fp.stat().st_size / 1e6
    print(f'  {aname}: {fp} ({size_mb:.0f} MB)')

## 4. Parse Contig IDs and Extract Per-Bin FASTAs

In [ ]:
def parse_contig_ids(contig_ids_str):
    """Parse contig_ids from CORAL — may be JSON array or comma-separated."""
    if pd.isna(contig_ids_str):
        return []
    s = str(contig_ids_str).strip()
    if s.startswith('['):
        return json.loads(s)
    return [c.strip() for c in s.split(',') if c.strip()]

# Test parsing
sample_ids = hq_bins.iloc[0]['contig_ids']
parsed = parse_contig_ids(sample_ids)
print(f'Sample bin has {len(parsed)} contigs')
print(f'First 3: {parsed[:3]}')

In [ ]:
def read_fasta_index(fasta_path):
    """Index a FASTA file: contig_id -> (offset, length). Works with gzipped files."""
    index = {}
    opener = gzip.open if str(fasta_path).endswith('.gz') else open
    with opener(fasta_path, 'rt') as f:
        current_id = None
        current_seqs = []
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if current_id is not None:
                    index[current_id] = '\n'.join(current_seqs)
                current_id = line[1:].split()[0]
                current_seqs = []
            else:
                current_seqs.append(line)
        if current_id is not None:
            index[current_id] = '\n'.join(current_seqs)
    return index

# Load and index all needed assemblies
needed_assembly_ids = set(hq_bins['assembly_id'].unique())
assembly_indices = {}

for aid in needed_assembly_ids:
    if aid not in assembly_fastas:
        print(f'WARNING: No FASTA for assembly {aid} — skipping its bins')
        continue
    fp = assembly_fastas[aid]
    aname = assemblies[assemblies['id'] == aid]['assembly_name'].values[0]
    print(f'Indexing {aname} ({fp})...')
    idx = read_fasta_index(fp)
    assembly_indices[aid] = idx
    print(f'  {len(idx)} contigs indexed')

In [ ]:
# Extract per-bin FASTAs
extraction_log = []

for _, row in hq_bins.iterrows():
    aid = row['assembly_id']
    bin_name = row['bin_name']
    
    if aid not in assembly_indices:
        extraction_log.append({'bin_name': bin_name, 'status': 'no_assembly', 'n_contigs': 0})
        continue
    
    idx = assembly_indices[aid]
    contig_ids = parse_contig_ids(row['contig_ids'])
    
    safe_name = bin_name.replace('/', '_').replace(' ', '_')
    out_path = MAG_DIR / f'{safe_name}.fasta'
    
    found = 0
    missing = []
    with open(out_path, 'w') as f:
        for cid in contig_ids:
            if cid in idx:
                f.write(f'>{cid}\n{idx[cid]}\n')
                found += 1
            else:
                missing.append(cid)
    
    status = 'ok' if not missing else f'partial_{len(missing)}_missing'
    extraction_log.append({
        'bin_name': bin_name, 'status': status, 
        'n_contigs': found, 'n_missing': len(missing),
        'fasta_path': str(out_path)
    })

log_df = pd.DataFrame(extraction_log)
print(f'Extraction summary:')
print(log_df['status'].value_counts())
print(f'\nTotal bins extracted: {(log_df["status"] != "no_assembly").sum()}')
print(f'Total contigs extracted: {log_df["n_contigs"].sum()}')

## 5. Summary & Export Metadata

In [ ]:
# Merge extraction log with bin metadata
mag_meta = hq_bins.merge(log_df, on='bin_name', how='left')
mag_meta_export = mag_meta[[
    'assembly_id', 'bin_name', 'completeness', 'contamination',
    'taxonomy', 'assembly_name', 'sample_name', 
    'status', 'n_contigs', 'fasta_path'
]]
mag_meta_export.to_csv(DATA_DIR / 'mag_metadata.tsv', sep='\t', index=False)

print(f'Saved metadata for {len(mag_meta_export)} QC-passing MAGs')
print(f'\nTaxonomy distribution (top 15):')
if mag_meta['taxonomy'].notna().any():
    tax_counts = mag_meta['taxonomy'].value_counts().head(15)
    for t, n in tax_counts.items():
        print(f'  {t}: {n}')
else:
    print('  No taxonomy annotations available in sdt_bin')

In [ ]:
# List FASTA files ready for annotation
fasta_files = sorted(MAG_DIR.glob('*.fasta'))
total_size_mb = sum(f.stat().st_size for f in fasta_files) / 1e6
print(f'{len(fasta_files)} FASTA files ready for bakta annotation')
print(f'Total size: {total_size_mb:.0f} MB')
print(f'\nNext step: NB02 submits these to CTS for bakta annotation')